## Replicação dados de Melanoma utilizando Pseudo-valores para S(t).

# Fórmula para os Pseudo-valores para S(t):
$$Y_{ij} = n\hat{S}_{KM}(t_j) - (n-1)\hat{S}_{KM}^{(-i)}(t_j)$$

In [3]:
library(survival)
library(smcure)

# Carregamento dos dados
data(e1684)
e1684 <- na.omit(e1684)

# Split treino/teste 
set.seed(2003)
n <- nrow(e1684)
train_ids <- sample(seq_len(n), size = floor(0.8 * n))

e1684$ID <- seq_len(n)
e1684$SET <- ifelse(e1684$ID %in% train_ids, "train", "test")

In [4]:
head(e1684)

,TRT,FAILTIME,FAILCENS,AGE,SEX,ID,SET
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>
1,1,1.15068,1,-11.0359437,0,1,train
2,1,0.62466,1,-5.1290437,0,2,train
3,0,1.89863,0,23.1859563,1,3,train
4,0,0.45479,1,11.1448563,1,4,train
5,0,2.09041,1,-13.3208437,0,5,train
6,1,9.38356,0,0.9421563,0,6,train


In [5]:
# Grade de tempos: quantis 10% a 90% + 95% dos tempos de evento
grade_tempos <- quantile(
  e1684$FAILTIME[e1684$FAILCENS > 0],
  probs = c((1:9) / 10, 0.95),
  names = FALSE
)

grade_tempos

[1] 0.130140 0.194520 0.294520 0.416440 0.550680 0.863010 1.120545 1.720550
 [9] 2.798630 3.652738

In [6]:
# Defição de função para calcular S_KM(t) em tempo específico

get_sobrevivencia <- function(fit, tempos) {
  summary(fit, times = tempos, extend = TRUE)$surv
}

In [7]:
# Função para cálculo dos Pseudo-valores na grade de tempos

calcula_pseudo <- function(time, status, tempos) {
  n <- length(time)
  m <- length(tempos)
  
  # Ajuste do modelo de Kaplan-Meier
  fit_completo <- survfit(Surv(time, status) ~ 1)

  # Estimativas de sobrevivência de KM para os tempos de interesse
  s_completo <- get_sobrevivencia(fit_completo, tempos)
  
  # Matriz de pseudo-valores
  pseudo <- matrix(NA, nrow = n, ncol = m)
  
  for (i in seq_len(n)) {
    # Ajuste do modelo de Kaplan-Meier sem a i-ésima observação
    fit_sem_i <- survfit(Surv(time[-i], status[-i]) ~ 1)

    # Estimativas de sobrevivência de KM sem a i-ésima observação para os tempos de interesse
    s_sem_i <- get_sobrevivencia(fit_sem_i, tempos)

    # Cálculo dos pseudo-valores para a i-ésima observação
    pseudo[i, ] <- n * s_completo - (n - 1) * s_sem_i
  }
  
  colnames(pseudo) <- paste0("t_", round(tempos, 6))
  pseudo
}

In [8]:
# Pseudo-valores da sobrevivencia total S(t):

pseudo_valores <- calcula_pseudo(
  time = e1684$FAILTIME,
  status = e1684$FAILCENS,
  tempos = grade_tempos
)

In [9]:
# Gera a matriz de entrada para o treinamento da rede

m <- length(grade_tempos)
entrada_rede <- data.frame(
  ID     = rep(e1684$ID, each = m),
  TRT    = rep(e1684$TRT, each = m),
  AGE    = rep(e1684$AGE, each = m),
  SEX    = rep(e1684$SEX, each = m),
  TIME   = rep(grade_tempos, times = n),
  PSEUDO = as.vector(t(pseudo_valores)),
  SET    = rep(e1684$SET, each = m)
)

In [10]:
head(entrada_rede, 20, digits = 12)

,ID,TRT,AGE,SEX,TIME,PSEUDO,SET
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,1,1,-11.035944,0,0.130140,1.000000000,train
2,1,1,-11.035944,0,0.194520,1.000000000,train
3,1,1,-11.035944,0,0.294520,1.000000000,train
4,1,1,-11.035944,0,0.416440,1.000000000,train
5,1,1,-11.035944,0,0.550680,1.000000000,train
6,1,1,-11.035944,0,0.863010,1.000439395,train
7,1,1,-11.035944,0,1.120545,1.001035717,train
8,1,1,-11.035944,0,1.720550,-0.003954554,train
9,1,1,-11.035944,0,2.798630,-0.003347867,train


In [11]:
# Debug psuedo-valor por indíduo
debug_pseudo_individuo <- function(df, tempos, i = 1) {
  n <- nrow(df)
  
  fit_full <- survfit(Surv(FAILTIME, FAILCENS) ~ 1, data = df)
  s_full <- summary(fit_full, times = tempos, extend = TRUE)$surv
  
  df_sem_i <- df[-i, , drop = FALSE]
  fit_sem_i <- survfit(Surv(FAILTIME, FAILCENS) ~ 1, data = df_sem_i)
  s_sem_i <- summary(fit_sem_i, times = tempos, extend = TRUE)$surv
  
  pseudo_i <- n * s_full - (n - 1) * s_sem_i
  
  primeiro_evento <- min(df$FAILTIME[df$FAILCENS == 1])
  
  dbg <- data.frame(
    tempo = tempos,
    S_full = s_full,
    S_sem_i = s_sem_i,
    pseudo_i = pseudo_i,
    diff_S = s_full - s_sem_i
  )
  
  cat("Individuo (linha i):", i, "\n")
  if ("ID" %in% names(df)) cat("ID:", df$ID[i], "\n")
  cat("Tempo do individuo:", df$FAILTIME[i], "| Status:", df$FAILCENS[i], "\n")
  print(dbg, digits = 12)
}

# Rodar debug para o individuo 1
debug_i1 <- debug_pseudo_individuo(e1684, grade_tempos, i = 1)

Individuo (linha i): 1 
ID: 1 
Tempo do individuo: 1.15068 | Status: 1 
       tempo         S_full        S_sem_i          pseudo_i             diff_S
1  0.1301400 0.929577464789 0.929328621908  1.00000000000000  0.000248842880605
2  0.1945200 0.855633802817 0.855123674912  1.00000000000009  0.000510127905241
3  0.2945200 0.792253521127 0.791519434629  1.00000000000009  0.000734086497786
4  0.4164400 0.721830985915 0.720848056537  1.00000000000020  0.000982929378391
5  0.5506800 0.651408450704 0.650176678445  1.00000000000023  0.001231772258996
6  0.8630100 0.584231646864 0.582760948108  1.00043939489072  0.001470698756277
7  1.1205450 0.516956487529 0.515245960218  1.00103571652772  0.001710527310951
8  1.7205500 0.446140530333 0.447730972327 -0.00395455401399 -0.001590441994158
9  2.7986300 0.377696020519 0.379042465352 -0.00334786734774 -0.001346444833450
10 3.6527375 0.341091436656 0.342307390168 -0.00302340724113 -0.001215953512006


In [14]:
# Salvar a entrada_rede e o e1684_bruto em dados/entrada para uso em python

dir_entrada <- file.path("..", "dados", "entrada")
dir.create(dir_entrada, recursive = TRUE, showWarnings = FALSE)

write.csv(entrada_rede, file.path(dir_entrada, "entrada_rede.csv"), row.names = FALSE)
write.csv(e1684, file.path(dir_entrada, "e1684_bruto.csv"), row.names = FALSE)